In [1]:
import numpy as np
import pandas as pd

In [5]:
NUMERIC_COLUMNS = ("balance", "duration", "campaign", "pdays", "previous")
CATEGORICAL_COLUMNS = (
    "job", "marital", "education", "default", "housing",
    "loan", "contact", "poutcome",
)
TRAIN_PATH = "train.csv"
columns = list(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS) + ["age"]

train = pd.read_csv(TRAIN_PATH, usecols=columns)
train.head(10)


,age,job,marital,education,default,balance,housing,loan,contact,duration,campaign,pdays,previous,poutcome
0,32,management,single,tertiary,no,0,no,no,cellular,68,6,-1,0,unknown
1,57,technician,married,secondary,no,783,no,no,cellular,324,1,183,2,failure
2,30,technician,single,secondary,no,961,yes,no,unknown,10,1,-1,0,unknown
3,28,technician,single,secondary,no,158,yes,yes,cellular,727,4,-1,0,unknown
4,58,entrepreneur,married,tertiary,no,33,no,no,cellular,96,1,-1,0,unknown
5,47,technician,married,primary,no,994,yes,no,unknown,121,5,-1,0,unknown
6,31,technician,divorced,secondary,no,545,yes,no,unknown,589,3,-1,0,unknown
7,39,technician,married,secondary,no,2684,yes,no,unknown,44,10,-1,0,unknown
8,40,admin.,married,primary,no,4943,yes,no,unknown,109,2,-1,0,unknown
9,57,admin.,divorced,secondary,no,3404,no,no,unknown,1110,1,-1,0,unknown


本笔记本对应根目录 `solution.py` 的预处理流程，用于分步查看数据。

训练阶段对应 `preprocess(train, fit=True)`：只从训练集记录均值、标准差和特征列名。
预测阶段对应 `preprocess(sample_df, fit=False)`：复用这些记录，不重新计算。

各个特征表都保留为 DataFrame，`age` 是标签，不进入特征表。


In [7]:
# 不使用 day 和 month
train = train[columns]

# 数值转换和缺失值填补，对应 preprocess() 的数值处理部分。
numeric = train[list(NUMERIC_COLUMNS)].apply(pd.to_numeric, errors="coerce")

# 对应 self.means：只在训练时记录均值；整列缺失时使用 0。
means = numeric.mean().fillna(0)
numeric = numeric.fillna(means)
numeric.head()


,balance,duration,campaign,pdays,previous
0,0,68,6,-1,0
1,783,324,1,183,2
2,961,10,1,-1,0
3,158,727,4,-1,0
4,33,96,1,-1,0


In [5]:
# 2. 在填补之后计算标准差，对应 self.stds。
# ddof=0 与 solution.py 一致；常数列的标准差改为 1，避免除以 0。
stds = numeric.std(ddof=0).replace(0, 1)

# 用新变量保留标准化结果，方便反复运行本单元格和比较填补前后的数据。
# numeric_scaled 对应 solution.py 标准化后的 numeric。
numeric_scaled = (numeric - means) / stds
numeric_scaled.head()


,balance,day,duration,campaign,pdays,previous
0,-0.423130,1.440606,-0.690095,1.257945,-0.302473,-0.222849
1,-0.147732,-0.741121,0.249726,-0.579661,2.078537,1.276194
2,-0.085125,-1.589570,-0.903023,-0.579661,-0.302473,-0.222849
3,-0.367558,1.804227,1.729211,0.522902,-0.302473,-0.222849
4,-0.411523,0.228536,-0.587302,-0.579661,-0.302473,-0.222849


In [6]:
# 3. 类别特征独热编码，参数与 solution.py 保持一致。
categorical = pd.get_dummies(
    train[list(CATEGORICAL_COLUMNS)],
    columns=list(CATEGORICAL_COLUMNS),
    prefix_sep="=",
    dtype=float,
)
categorical.head()


,job=admin.,job=blue-collar,job=entrepreneur,job=housemaid,job=management,job=retired,job=self-employed,job=services,job=student,job=technician,...,month=jun,month=mar,month=may,month=nov,month=oct,month=sep,poutcome=failure,poutcome=other,poutcome=success,poutcome=unknown
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
# 4. 拼接特征并记录训练列名，对应 fit=True 分支中的 self.feature_names。
features = pd.concat([numeric_scaled, categorical], axis=1)
feature_names = features.columns.tolist()
features = features.reindex(columns=feature_names, fill_value=0.0)
features.head()


,balance,day,duration,campaign,pdays,previous,job=admin.,job=blue-collar,job=entrepreneur,job=housemaid,...,month=jun,month=mar,month=may,month=nov,month=oct,month=sep,poutcome=failure,poutcome=other,poutcome=success,poutcome=unknown
0,-0.423130,1.440606,-0.690095,1.257945,-0.302473,-0.222849,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.147732,-0.741121,0.249726,-0.579661,2.078537,1.276194,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,-0.085125,-1.589570,-0.903023,-0.579661,-0.302473,-0.222849,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,-0.367558,1.804227,1.729211,0.522902,-0.302473,-0.222849,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.411523,0.228536,-0.587302,-0.579661,-0.302473,-0.222849,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [8]:
# features 是 DataFrame，可以直接按列名选择和检查。
features[["balance", "duration"]].head()


,balance,duration
0,-0.423130,-0.690095
1,-0.147732,0.249726
2,-0.085125,-0.903023
3,-0.367558,1.729211
4,-0.411523,-0.587302


下面用一条训练记录演示预测阶段的预处理，只处理特征，不训练模型。

这部分对应 `forward()` 中的字典转 DataFrame，以及 `preprocess(..., fit=False)`。
这里沿用训练时的 `means`、`stds` 和 `feature_names`。


In [9]:
# 5. 模拟 forward() 接收到的一条样本，不提供 age。
sample = train.drop(columns=["age"]).iloc[0].to_dict()
sample_df = pd.DataFrame([sample])

# 复用训练均值填补，复用训练均值和标准差进行标准化。
sample_numeric = sample_df[list(NUMERIC_COLUMNS)].apply(pd.to_numeric, errors="coerce")
sample_numeric = sample_numeric.fillna(means)
sample_numeric = (sample_numeric - means) / stds

sample_categorical = pd.get_dummies(
    sample_df[list(CATEGORICAL_COLUMNS)],
    columns=list(CATEGORICAL_COLUMNS),
    prefix_sep="=",
    dtype=float,
)
sample_features = pd.concat([sample_numeric, sample_categorical], axis=1)

# 缺少的类别列补 0，未知类别列移除；列名和顺序都沿用训练时的记录。
sample_features = sample_features.reindex(columns=feature_names, fill_value=0.0)
sample_features


,balance,day,duration,campaign,pdays,previous,job=admin.,job=blue-collar,job=entrepreneur,job=housemaid,...,month=jun,month=mar,month=may,month=nov,month=oct,month=sep,poutcome=failure,poutcome=other,poutcome=success,poutcome=unknown
0,-0.42313,1.440606,-0.690095,1.257945,-0.302473,-0.222849,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [10]:
# 同一条记录，无论批量处理还是单条处理，结果都应相同。
pd.testing.assert_frame_equal(
    sample_features,
    features.iloc[[0]].reset_index(drop=True),
)
assert features.columns.tolist() == sample_features.columns.tolist()
assert "age" not in features.columns and "id" not in features.columns
print("Batch and single-sample preprocessing are consistent.")


Batch and single-sample preprocessing are consistent.


变量对应关系：

| 本笔记本 | 根目录 solution.py |
|---|---|
| means | self.means |
| stds | self.stds |
| numeric_scaled | 标准化后的 numeric |
| feature_names | self.feature_names |
| features | preprocess(train, fit=True) 的返回结果 |
| sample_features | preprocess(sample_df, fit=False) 的返回结果 |

填补方法与根目录代码一致，使用训练均值。预处理没有增加中位数填补或额外的数值清洗。

实际训练时，根目录的 Model 在内部把特征表转换成 NumPy 数组，再进行矩阵运算。
修改预处理方案后，应从头运行本笔记本，并同步修改 solution.py 中的对应步骤。
